# 09. SMART Barangs & Lots Fine-Grained Subcategory Consolidation
### Modular, Subcategory-by-Subcategory Deduplication & Master Data Synchronization
---
This notebook serves as the final refinement stage of the AMS-to-SMART data migration:
1. **Granular Normalization**: Iterates through individual subcategories to resolve messy specifications, embedded location strings, and syntax anomalies.
2. **Modular Architecture**: Each subcategory has its own dedicated, self-contained execution cell.
3. **Master Re-Synchronization**: After updating any subcategory, `sync_consolidated_masters()` automatically aggregates all 111 subcategories, re-indexes sequential IDs, re-links units, and re-exports all staging and final deliverables.
4. **Data Integrity Assured**: Every execution re-validates 100% of global referential and schema assertions.

In [1]:
import os
import sys
import json
import re
from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. Dynamic Path Resolution
# ---------------------------------------------------------
def get_data_dir():
    candidates = [
        Path('/home/jovyan/data'),                      # Docker container path
        Path('data'),                                  # Repository root
        Path('../data'),                               # Notebooks directory
        Path('/home/ran/Devs/ams-smart-migration/data') # Host absolute path
    ]
    for p in candidates:
        if p.exists() and (p / 'smart').exists():
            return p.resolve()
    return Path('data').resolve()

DATA_DIR = get_data_dir()
STAGING_ASSETS_DIR = DATA_DIR / 'smart' / 'staging' / 'assets'
STAGING_UNITS_DIR = DATA_DIR / 'smart' / 'staging' / 'units'
STAGING_BARANGS_DIR = DATA_DIR / 'smart' / 'staging' / 'barangs'
STAGING_LOTS_DIR = DATA_DIR / 'smart' / 'staging' / 'lots'

FINAL_BARANG_DIR = DATA_DIR / 'smart' / 'final' / 'barang'
FINAL_LOT_DIR = DATA_DIR / 'smart' / 'final' / 'lot'
FINAL_UNIT_DIR = DATA_DIR / 'smart' / 'final' / 'unit'

FINAL_SUBCATEGORY_PATH = DATA_DIR / 'smart' / 'final' / 'subcategory' / 'subcategories.csv'
FINAL_BRAND_PATH = DATA_DIR / 'smart' / 'final' / 'brand' / 'brands.csv'
FINAL_VENDOR_PATH = DATA_DIR / 'smart' / 'final' / 'vendor' / 'vendors.csv'

STAGING_BRAND_MAP_PATH = DATA_DIR / 'smart' / 'staging' / 'brand' / 'brand_ams_mapping.csv'
STAGING_VENDOR_MAP_PATH = DATA_DIR / 'smart' / 'staging' / 'vendor' / 'vendor_ams_mapping.csv'
UNITS_MANIFEST_PATH = STAGING_UNITS_DIR / '_units_manifest.json'

for d in [STAGING_BARANGS_DIR, STAGING_LOTS_DIR, FINAL_BARANG_DIR, FINAL_LOT_DIR, FINAL_UNIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 2. Load Master Lookups
# ---------------------------------------------------------
df_sub = pd.read_csv(FINAL_SUBCATEGORY_PATH, dtype=str)
sub_map = dict(zip(df_sub['code'], df_sub['id']))

df_brand = pd.read_csv(FINAL_BRAND_PATH, dtype=str)
brand_name_to_id = {r['name'].strip().upper(): str(r['id']).strip() for _, r in df_brand.iterrows() if pd.notna(r['name'])}

df_bmap = pd.read_csv(STAGING_BRAND_MAP_PATH, dtype=str)
ams_brand_to_id = {}
for _, r in df_bmap.iterrows():
    if pd.notna(r.get('ams_brand_name')):
        ams_brand_to_id[r['ams_brand_name'].strip().upper()] = str(r['smart_brand_id']).strip()

brand_aliases = {
    'LIONSTAR': brand_name_to_id.get('LION STAR', '125'),
    'LION STAR': brand_name_to_id.get('LION STAR', '125'),
    'PONGS': brand_name_to_id.get('PONGS HOME', '79'),
    'PONGS HOME': brand_name_to_id.get('PONGS HOME', '79'),
    'SYLYART': brand_name_to_id.get('SYLY ART', '130'),
    'SYLU ART': brand_name_to_id.get('SYLY ART', '130'),
    'SOFA': '123',
    'NO BRAND': '123',
    'CHITOSE': '173',
    'ICA': '174',
    'STELA': '175',
    'STELLA': '175',
    'MORELLY': '176',
}

df_vmap = pd.read_csv(STAGING_VENDOR_MAP_PATH, dtype=str)
vendor_map = {}
for _, r in df_vmap.iterrows():
    v_id = str(r['ams_id']).strip()
    if v_id.endswith('.0'): v_id = v_id[:-2]
    vendor_map[v_id] = str(r['smart_vendor_id']).strip()

def resolve_brand_id(raw_brand):
    if pd.isna(raw_brand) or not str(raw_brand).strip():
        return '123'
    b = str(raw_brand).strip().upper()
    if b in brand_aliases:
        return brand_aliases[b]
    if b in brand_name_to_id:
        return brand_name_to_id[b]
    if b in ams_brand_to_id:
        return ams_brand_to_id[b]
    return '123'

def resolve_vendor_id(raw_vendor):
    if pd.isna(raw_vendor) or str(raw_vendor).strip() in ('', 'NULL', 'null', 'None'):
        return None
    v = str(raw_vendor).strip()
    if v.endswith('.0'): v = v[:-2]
    return vendor_map.get(v, None)

def resolve_organizer_id(raw_org):
    return '2' if str(raw_org).strip().upper() == 'ICT' else '1'

def clean_text(s):
    if pd.isna(s) or s is None:
        return ''
    return re.sub(r'\s+', ' ', str(s).replace('\r', ' ').replace('\n', ' ')).strip()

# ---------------------------------------------------------
# 3. Master Re-Synchronization Engine
# ---------------------------------------------------------
def escape_sql(val):
    if val is None or pd.isna(val):
        return 'NULL'
    s = str(val).replace("'", "''")
    return f"N'{s}'"

def format_num_sql(val):
    if val is None or pd.isna(val) or str(val).strip() in ('', 'None', 'nan', 'NULL'):
        return 'NULL'
    try:
        f_val = float(val)
        return str(int(f_val)) if f_val.is_integer() else str(f_val)
    except (ValueError, TypeError):
        return str(val)

def sync_consolidated_masters():
    with open(UNITS_MANIFEST_PATH, 'r', encoding='utf-8') as f:
        manifest = json.load(f)
        
    all_barangs = []
    all_lots = []
    all_units = []
    
    current_id = 1
    barangs_manifest = {'subcategories': {}, 'total_barangs': 0}
    lots_manifest = {'subcategories': {}, 'total_lots': 0}
    units_manifest = {'subcategories': {}, 'total_units': 0}
    
    for sub_code, meta in sorted(manifest['subcategories'].items()):
        b_file = STAGING_BARANGS_DIR / meta['file']
        l_file = STAGING_LOTS_DIR / meta['file']
        u_file = STAGING_UNITS_DIR / meta['file']
        
        if not b_file.exists() or meta['row_count'] == 0:
            continue
            
        df_b = pd.read_csv(b_file, dtype=str)
        df_l = pd.read_csv(l_file, dtype=str)
        df_u = pd.read_csv(u_file, dtype=str)
        
        # Re-index IDs sequentially for this partition
        old_b_to_new_id = {}
        for idx in range(len(df_b)):
            old_b_to_new_id[str(df_b.at[idx, 'id'])] = current_id
            df_b.at[idx, 'id'] = current_id
            df_l.at[idx, 'id'] = current_id
            df_l.at[idx, 'barang_id'] = current_id
            current_id += 1
            
        # Save re-indexed partition files
        df_b.to_csv(b_file, index=False)
        df_l.to_csv(l_file, index=False)
        
        # Update units lot_id mapping
        if 'lot_id' in df_u.columns:
            df_u['lot_id'] = df_u['lot_id'].map(lambda x: old_b_to_new_id.get(str(x), x))
            for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
                if c in df_u.columns:
                    df_u[c] = pd.to_numeric(df_u[c], errors='coerce').astype('Int64')
            df_u.to_csv(u_file, index=False)
            
        all_barangs.append(df_b)
        all_lots.append(df_l)
        all_units.append(df_u)
        
        barangs_manifest['subcategories'][sub_code] = {'file': meta['file'], 'row_count': len(df_b)}
        lots_manifest['subcategories'][sub_code] = {'file': meta['file'], 'row_count': len(df_l)}
        units_manifest['subcategories'][sub_code] = {'file': meta['file'], 'row_count': len(df_u)}
        
    df_barangs_master = pd.concat(all_barangs, ignore_index=True)
    df_lots_master = pd.concat(all_lots, ignore_index=True)
    df_units_master = pd.concat(all_units, ignore_index=True)
    
    barangs_manifest['total_barangs'] = len(df_barangs_master)
    lots_manifest['total_lots'] = len(df_lots_master)
    units_manifest['total_units'] = len(df_units_master)
    
    # 1. Staging Masters
    df_barangs_master.to_csv(STAGING_BARANGS_DIR / 'barangs.csv', index=False)
    with open(STAGING_BARANGS_DIR / '_barangs_manifest.json', 'w', encoding='utf-8') as f:
        json.dump(barangs_manifest, f, indent=2)
        
    df_lots_master.to_csv(STAGING_LOTS_DIR / 'lots.csv', index=False)
    with open(STAGING_LOTS_DIR / '_lots_manifest.json', 'w', encoding='utf-8') as f:
        json.dump(lots_manifest, f, indent=2)
        
    df_units_master.to_csv(STAGING_UNITS_DIR / 'units.csv', index=False)
    with open(STAGING_UNITS_DIR / '_units_manifest.json', 'w', encoding='utf-8') as f:
        json.dump(units_manifest, f, indent=2)
        
    # 2. Final Production Masters
    df_barangs_master.to_csv(FINAL_BARANG_DIR / 'barangs.csv', index=False)
    df_lots_master.to_csv(FINAL_LOT_DIR / 'lots.csv', index=False)
    
    db_ordered_cols = [
        'id', 'number', 'legacy_number', 'lot_id', 'location_id',
        'status', 'condition', 'price', 'image_url', 'vehicle_registration',
        'burden', 'project_id', 'created_at', 'updated_at', 'type',
        'classification', 'vendor_id'
    ]
    df_units_final = df_units_master[[
        'id', 'number', 'legacy_number', 'lot_id', 'location_id',
        'status', 'condition', 'price', 'image_url', 'burden',
        'project_id', 'created_at', 'updated_at', 'type',
        'classification', 'vendor_id'
    ]].copy()
    df_units_final['vehicle_registration'] = None
    df_units_final = df_units_final[db_ordered_cols]
    for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
        df_units_final[c] = pd.to_numeric(df_units_final[c], errors='coerce').astype('Int64')
    df_units_final.to_csv(FINAL_UNIT_DIR / 'units.csv', index=False)
    
    # 3. SQL Insert Scripts
    with open(FINAL_BARANG_DIR / 'barangs.sql', 'w', encoding='utf-8') as f:
        f.write('SET IDENTITY_INSERT [barangs] ON;\nGO\n\n')
        for _, r in df_barangs_master.iterrows():
            f.write(f"INSERT INTO [barangs] ([id], [number], [subcategory_id], [brand_id], [uom_id], [name], [specification], [min_stock_threshold], [image_url], [last_restock_at], [created_at], [updated_at]) VALUES ({r['id']}, {escape_sql(r['number'])}, {r['subcategory_id']}, {r['brand_id']}, {r['uom_id']}, {escape_sql(r['name'])}, {escape_sql(r['specification'])}, {format_num_sql(r['min_stock_threshold'])}, {escape_sql(r['image_url'])}, {escape_sql(r['last_restock_at'])}, '{r['created_at']}', '{r['updated_at']}');\n")
        f.write('\nGO\nSET IDENTITY_INSERT [barangs] OFF;\nGO\n')
        
    with open(FINAL_LOT_DIR / 'lots.sql', 'w', encoding='utf-8') as f:
        f.write('SET IDENTITY_INSERT [lots] ON;\nGO\n\n')
        for _, r in df_lots_master.iterrows():
            f.write(f"INSERT INTO [lots] ([id], [number], [barang_id], [organizer_id], [vendor_id], [legacy_vendor_id], [location_id], [initial_quantity], [current_quantity], [po_number], [date_of_receipt], [unit_price], [image_url], [burden], [project_id], [created_at], [updated_at]) VALUES ({r['id']}, {escape_sql(r['number'])}, {r['barang_id']}, {r['organizer_id']}, NULL, NULL, NULL, NULL, NULL, NULL, '{r['date_of_receipt']}', NULL, {escape_sql(r['image_url'])}, NULL, NULL, '{r['created_at']}', '{r['updated_at']}');\n")
        f.write('\nGO\nSET IDENTITY_INSERT [lots] OFF;\nGO\n')
        
    with open(FINAL_UNIT_DIR / 'units.sql', 'w', encoding='utf-8') as f:
        f.write('SET IDENTITY_INSERT [units] ON;\nGO\n\n')
        for _, r in df_units_final.iterrows():
            f.write(f"INSERT INTO [units] ([id], [number], [legacy_number], [lot_id], [location_id], [status], [condition], [price], [image_url], [vehicle_registration], [burden], [project_id], [created_at], [updated_at], [type], [classification], [vendor_id]) VALUES ({r['id']}, {escape_sql(r['number'])}, {escape_sql(r['legacy_number'])}, {r['lot_id']}, {r['location_id']}, {escape_sql(r['status'])}, {escape_sql(r['condition'])}, {format_num_sql(r['price'])}, {escape_sql(r['image_url'])}, {escape_sql(r['vehicle_registration'])}, {escape_sql(r['burden'])}, {format_num_sql(r['project_id'])}, '{r['created_at']}', '{r['updated_at']}', {escape_sql(r['type'])}, {escape_sql(r['classification'])}, {format_num_sql(r['vendor_id'])});\n")
        f.write('\nGO\nSET IDENTITY_INSERT [units] OFF;\nGO\n')
        
    # Global Quality Assertions
    assert len(df_units_master) == 6943, f'Expected 6,943 units, got {len(df_units_master)}'
    assert len(df_barangs_master) == len(df_lots_master)
    assert (df_barangs_master['id'].astype(int) == list(range(1, len(df_barangs_master) + 1))).all()
    assert (df_lots_master['id'].astype(int) == list(range(1, len(df_lots_master) + 1))).all()
    assert df_barangs_master['number'].is_unique
    assert df_lots_master['number'].is_unique
    assert df_barangs_master['number'].str.len().max() <= 14
    assert df_lots_master['number'].str.len().max() <= 26
    
    print(f' Master Sync Complete: {len(df_barangs_master):,} barangs, {len(df_lots_master):,} lots, {len(df_units_master):,} units.')
    return df_barangs_master, df_lots_master, df_units_master

print('Infrastructure setup and sync engine initialized successfully.')


Infrastructure setup and sync engine initialized successfully.


---
## Subcategory Consolidation: `ACC-AP` (Alat Pemadam Api Ringan / APAR)

### Business Context & Cleaning Rules:
In `ACC-AP`, asset entries originally contained location details typed directly into `spesifikasi` (*"Pos Security Samping"*, *"Depan Resepsionist"*, *"Samping Lift"*), decimal typing variations (`3,5` vs `3.5`), and redundant prefix tags (*"Jenis Apar"*, *"Apar Jenis"*).

**Consolidation Transformations Applied:**
1. **Strip Location Notes**: Physical locations belong in `units.location_id`, not in the catalog definition.
2. **Normalize Decimals**: Unify decimal comma `3,5` $\rightarrow$ `3.5`.
3. **Strip Prefixes**: Remove leading `Jenis Apar`, `Apar Jenis`, `Jenis`.
4. **Standardize Gases**: Standardize `CO 2` $\rightarrow$ `CO2`.
5. **Outcome**: Consolidates **32 fragmented items into exactly 12 standardized catalog barangs** across all 53 units.

In [2]:
# ---------------------------------------------------------
# Subcategory: ACC-AP Consolidation Cell
# ---------------------------------------------------------
SUB_CODE = 'ACC-AP'

def normalize_apar_spec(spec):
    if pd.isna(spec) or not str(spec).strip() or str(spec).strip().lower() in ('nan', 'none', 'null'):
        return None
    s = str(spec).replace('\r', ' ').replace('\n', ' ')
    s = re.sub(r'\s+', ' ', s).strip()
    
    # 1. Strip leading prefixes
    s = re.sub(r'^(jenis\s+apar\s+|apar\s+jenis\s+|jenis\s+)', '', s, flags=re.IGNORECASE)
    
    # 2. Normalize decimal commas to periods
    s = re.sub(r'(\d+),(\d+)', r'\1.\2', s)
    
    # 3. Standardize CO 2 -> CO2
    s = re.sub(r'co\s*2', 'CO2', s, flags=re.IGNORECASE)
    
    # 4. Extract core type and capacity, stripping location notes
    m_dc = re.search(r'dry\s+chemical\s*(\d+(?:\.\d+)?)\s*(?:kg)?', s, flags=re.IGNORECASE)
    if m_dc:
        return f'Dry Chemical {m_dc.group(1)} kg'
        
    m_co2 = re.search(r'co2\s*(\d+(?:\.\d+)?)\s*(?:kg)?', s, flags=re.IGNORECASE)
    if m_co2:
        return f'CO2 {m_co2.group(1)} kg'
        
    if 'powder' in s.lower() or 'abc' in s.lower():
        m_w = re.search(r'(\d+(?:\.\d+)?)\s*kg', s, flags=re.IGNORECASE)
        w = m_w.group(1) if m_w else '6'
        return f'Powder ABC {w} kg'
        
    return s

# Load partition datasets
df_ap_assets = pd.read_csv(STAGING_ASSETS_DIR / f'{SUB_CODE}.csv', dtype=str)
df_ap_units = pd.read_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', dtype=str)

df_ap_merged = df_ap_assets.merge(
    df_ap_units[['ams_asset_id', 'image_url', 'created_at', 'updated_at']],
    left_on='id',
    right_on='ams_asset_id',
    suffixes=('_asset', '_unit')
)

# Group by (brand_id, name, normalized_spec)
groups = defaultdict(list)
for _, r in df_ap_merged.iterrows():
    b_id = resolve_brand_id(r.get('brand'))
    spec_norm = normalize_apar_spec(r.get('spesifikasi'))
    spec_key = spec_norm.upper() if spec_norm else ''
    groups[(b_id, 'APAR', spec_key)].append(r)

# Sort groups deterministically
sorted_groups = sorted(groups.items(), key=lambda item: int(float(item[1][0]['id'])))

sub_barangs = []
sub_lots = []
unit_lot_map = {}
seq_counter = 1
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S.000')

for (b_id, name_val, spec_key), group_rows in sorted_groups:
    barang_num = f'{SUB_CODE}-{seq_counter:04d}'
    lot_num = f'LOT-0001-26-{barang_num}'
    display_spec = normalize_apar_spec(group_rows[0].get('spesifikasi'))
    
    # First non-null image
    first_img = None
    for r in group_rows:
        img = r.get('image_url') or r.get('image_url_unit')
        if pd.notna(img) and str(img).strip():
            first_img = str(img).strip()
            break
            
    created_ats = [str(r.get('created_at') or r.get('created_at_unit')) for r in group_rows if pd.notna(r.get('created_at') or r.get('created_at_unit')) and str(r.get('created_at') or r.get('created_at_unit')).strip()]
    updated_ats = [str(r.get('updated_at') or r.get('updated_at_unit')) for r in group_rows if pd.notna(r.get('updated_at') or r.get('updated_at_unit')) and str(r.get('updated_at') or r.get('updated_at_unit')).strip()]
    
    oldest_created = min(created_ats) if created_ats else now_str
    newest_updated = max(updated_ats) if updated_ats else now_str
    org_id = resolve_organizer_id(group_rows[0].get('organizer'))
    
    b_dict = {
        'id': seq_counter,
        'number': barang_num,
        'subcategory_id': sub_map[SUB_CODE],
        'brand_id': b_id,
        'uom_id': 1,
        'name': 'APAR',
        'specification': display_spec,
        'min_stock_threshold': None,
        'image_url': first_img,
        'last_restock_at': None,
        'created_at': oldest_created,
        'updated_at': newest_updated
    }
    
    l_dict = {
        'id': seq_counter,
        'number': lot_num,
        'barang_id': seq_counter,
        'organizer_id': int(org_id),
        'vendor_id': None,
        'legacy_vendor_id': None,
        'location_id': None,
        'initial_quantity': None,
        'current_quantity': None,
        'po_number': None,
        'date_of_receipt': now_str,
        'unit_price': None,
        'image_url': first_img,
        'burden': None,
        'project_id': None,
        'created_at': now_str,
        'updated_at': now_str
    }
    
    sub_barangs.append(b_dict)
    sub_lots.append(l_dict)
    
    for r in group_rows:
        unit_lot_map[str(r['id']).strip()] = seq_counter
        
    seq_counter += 1

# Save local partition files
df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f'{SUB_CODE}.csv', index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f'{SUB_CODE}.csv', index=False)

# Update units lot_id in staging
for idx in range(len(df_ap_units)):
    ams_id = str(df_ap_units.at[idx, 'ams_asset_id']).strip()
    if ams_id in unit_lot_map:
        df_ap_units.at[idx, 'lot_id'] = unit_lot_map[ams_id]

for c in ['id', 'lot_id', 'location_id', 'project_id', 'vendor_id']:
    if c in df_ap_units.columns:
        df_ap_units[c] = pd.to_numeric(df_ap_units[c], errors='coerce').astype('Int64')
df_ap_units.to_csv(STAGING_UNITS_DIR / f'{SUB_CODE}.csv', index=False)

# Synchronize global masters & SQL scripts
df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

# Local Assertions
assert len(df_sub_barangs) == 12, f'Expected 12 barangs, got {len(df_sub_barangs)}'
assert len(df_ap_units) == 53, f'Expected 53 units, got {len(df_ap_units)}'

print(f'\n[SUCCESS] Consolidated ACC-AP from 32 barangs down to {len(df_sub_barangs)} barangs across 53 units.')

# Display the consolidated items table
ap_display = df_sub_barangs[['id', 'number', 'brand_id', 'name', 'specification']].copy()
brand_id_to_name = dict(zip(df_brand['id'], df_brand['name']))
ap_display['brand'] = ap_display['brand_id'].astype(str).map(brand_id_to_name)
ap_unit_counts = df_ap_units.groupby('lot_id').size().to_dict()
ap_display['unit_count'] = ap_display['id'].astype(int).map(ap_unit_counts)
display(ap_display[['id', 'number', 'brand', 'name', 'specification', 'unit_count']])


 Master Sync Complete: 1,443 barangs, 1,443 lots, 6,943 units.

[SUCCESS] Consolidated ACC-AP from 32 barangs down to 12 barangs across 53 units.


,id,number,brand,name,specification,unit_count
0,1,ACC-AP-0001,Appron,APAR,Dry Chemical 3.5 kg,3
1,2,ACC-AP-0002,Firex,APAR,Dry Chemical 3.5 kg,15
2,3,ACC-AP-0003,Appron,APAR,Dry Chemical 2.5 kg,2
3,4,ACC-AP-0004,Firex,APAR,Dry Chemical 2.5 kg,1
4,5,ACC-AP-0005,Firex,APAR,Dry Chemical 3 kg,9
5,6,ACC-AP-0006,Firex,APAR,CO2 7 kg,2
6,7,ACC-AP-0007,Firex,APAR,Dry Chemical 6 kg,3
7,8,ACC-AP-0008,Firex,APAR,CO2 6 kg,2
8,9,ACC-AP-0009,Firex,APAR,CO2 5 kg,2
9,10,ACC-AP-0010,Firex,APAR,CO2 25 kg,1


---
## 2. Subcategory ACC-BT: Braket TV Consolidation
### Normalization Rules applied:
1. **Specification**: Ignore redundant 'Braket Besi' (all brackets are steel; canonical spec becomes None).
2. **Word Order**: Canonicalize name variations like 'Braket Standing TV' vs 'Braket TV Standing' into 'Braket TV Standing'.

In [3]:
# =====================================================================
# Subcategory ACC-BT: Braket TV Deduplication & Consolidation
# =====================================================================
SUB_CODE = "ACC-BT"
print(f"Consolidating {SUB_CODE}...")

def normalize_bt_name(raw_name):
    if not raw_name or pd.isna(raw_name):
        return "Braket TV Standing"
    words = set(re.findall(r"\b\w+\b", str(raw_name).strip().upper()))
    if {"BRAKET", "TV", "STANDING"}.issubset(words):
        return "Braket TV Standing"
    return str(raw_name).strip().title()

def normalize_bt_spec(raw_spec):
    if not raw_spec or pd.isna(raw_spec):
        return None
    s = str(raw_spec).strip()
    if s.upper() in ["BRAKET BESI", "BESI", "NULL", "NONE"]:
        return None
    return s if s else None

# 1. Load source partitions
df_bt_assets = pd.read_csv(STAGING_ASSETS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_bt_units = pd.read_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", dtype=str)

df_bt_merged = df_bt_assets.merge(
    df_bt_units[["ams_asset_id", "image_url", "created_at", "updated_at"]],
    left_on="id",
    right_on="ams_asset_id",
    how="left",
    suffixes=("", "_unit")
)

# 2. Group by (brand_id, normalized_name, normalized_spec)
groups = defaultdict(list)
for _, r in df_bt_merged.iterrows():
    b_id = resolve_brand_id(r.get("brand"))
    name_norm = normalize_bt_name(r.get("name"))
    spec_norm = normalize_bt_spec(r.get("spesifikasi"))
    spec_key = spec_norm.upper() if spec_norm else ""
    groups[(b_id, name_norm, spec_key)].append(r)

# Deterministic sorting
sorted_groups = sorted(groups.items(), key=lambda item: int(float(item[1][0]["id"])))

sub_barangs = []
sub_lots = []
unit_lot_map = {}
seq_counter = 1
now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S.000")

for (b_id, name_val, spec_key), group_rows in sorted_groups:
    barang_num = f"{SUB_CODE}-{seq_counter:04d}"
    lot_num = f"LOT-0001-26-{barang_num}"
    display_spec = normalize_bt_spec(group_rows[0].get("spesifikasi"))
    
    # First non-null image
    first_img = None
    for r in group_rows:
        img = r.get("image_url") or r.get("image_url_unit")
        if pd.notna(img) and str(img).strip():
            first_img = str(img).strip()
            break
            
    created_ats = [str(r.get("created_at") or r.get("created_at_unit")) for r in group_rows if pd.notna(r.get("created_at") or r.get("created_at_unit")) and str(r.get("created_at") or r.get("created_at_unit")).strip()]
    updated_ats = [str(r.get("updated_at") or r.get("updated_at_unit")) for r in group_rows if pd.notna(r.get("updated_at") or r.get("updated_at_unit")) and str(r.get("updated_at") or r.get("updated_at_unit")).strip()]
    
    oldest_created = min(created_ats) if created_ats else now_str
    newest_updated = max(updated_ats) if updated_ats else now_str
    org_id = resolve_organizer_id(group_rows[0].get("organizer"))
    
    b_dict = {
        "id": seq_counter,
        "number": barang_num,
        "subcategory_id": sub_map[SUB_CODE],
        "brand_id": b_id,
        "uom_id": 1,
        "name": name_val,
        "specification": display_spec,
        "min_stock_threshold": None,
        "image_url": first_img,
        "last_restock_at": None,
        "created_at": oldest_created,
        "updated_at": newest_updated
    }
    
    l_dict = {
        "id": seq_counter,
        "number": lot_num,
        "barang_id": seq_counter,
        "organizer_id": int(org_id),
        "vendor_id": None,
        "legacy_vendor_id": None,
        "location_id": None,
        "initial_quantity": None,
        "current_quantity": None,
        "po_number": None,
        "date_of_receipt": now_str,
        "unit_price": None,
        "image_url": first_img,
        "burden": None,
        "project_id": None,
        "created_at": now_str,
        "updated_at": now_str
    }
    
    sub_barangs.append(b_dict)
    sub_lots.append(l_dict)
    
    for r in group_rows:
        unit_lot_map[str(r["id"]).strip()] = seq_counter
        
    seq_counter += 1

# 3. Save local partition files
df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f"{SUB_CODE}.csv", index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f"{SUB_CODE}.csv", index=False)

# Update units lot_id in staging
for idx in range(len(df_bt_units)):
    ams_id = str(df_bt_units.at[idx, "ams_asset_id"]).strip()
    if ams_id in unit_lot_map:
        df_bt_units.at[idx, "lot_id"] = unit_lot_map[ams_id]

for c in ["id", "lot_id", "location_id", "project_id", "vendor_id"]:
    if c in df_bt_units.columns:
        df_bt_units[c] = pd.to_numeric(df_bt_units[c], errors="coerce").astype("Int64")
df_bt_units.to_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", index=False)

# 4. Global synchronization
df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

# 5. Local assertions
assert len(df_sub_barangs) == 3, f"Expected 3 barangs for ACC-BT, got {len(df_sub_barangs)}"
assert len(df_bt_units) == 7, f"Expected 7 units for ACC-BT, got {len(df_bt_units)}"
print(f"\n[SUCCESS] Consolidated ACC-BT from 5 barangs down to {len(df_sub_barangs)} barangs across 7 units.")

# Display items summary table
bt_display = df_sub_barangs[["id", "number", "brand_id", "name", "specification"]].copy()
brand_id_to_name = dict(zip(df_brand["id"], df_brand["name"]))
bt_display["brand"] = bt_display["brand_id"].astype(str).map(brand_id_to_name)
bt_unit_counts = df_bt_units.groupby("lot_id").size().to_dict()
bt_display["unit_count"] = bt_display["id"].astype(int).map(bt_unit_counts)
display(bt_display[["id", "number", "brand", "name", "specification", "unit_count"]])


Consolidating ACC-BT...
 Master Sync Complete: 1,443 barangs, 1,443 lots, 6,943 units.

[SUCCESS] Consolidated ACC-BT from 5 barangs down to 3 barangs across 7 units.


,id,number,brand,name,specification,unit_count
0,1,ACC-BT-0001,Panasonic,Braket TV Standing,None,2
1,2,ACC-BT-0002,No Brand,Braket TV Standing,None,1
2,3,ACC-BT-0003,North Bayou,Braket TV Standing,None,4


---
## 3. Subcategory ACC-CM: Cermin Cembung Consolidation
### Normalization Rules applied:
1. Combine both items into one canonical item sharing Brand ID 123 (No Brand).
2. **Name**: 
3. **Specification**: 

In [4]:
# =====================================================================
# Subcategory ACC-CM: Cermin Cembung Deduplication & Consolidation
# =====================================================================
SUB_CODE = "ACC-CM"
print(f"Consolidating {SUB_CODE}...")

df_cm_assets = pd.read_csv(STAGING_ASSETS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_cm_units = pd.read_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", dtype=str)

df_cm_merged = df_cm_assets.merge(
    df_cm_units[["ams_asset_id", "image_url", "created_at", "updated_at"]],
    left_on="id",
    right_on="ams_asset_id",
    how="left",
    suffixes=("", "_unit")
)

first_img = None
for _, r in df_cm_merged.iterrows():
    img = r.get("image_url") or r.get("image_url_unit")
    if pd.notna(img) and str(img).strip():
        first_img = str(img).strip()
        break

created_ats = [str(r.get("created_at") or r.get("created_at_unit")) for _, r in df_cm_merged.iterrows() if pd.notna(r.get("created_at") or r.get("created_at_unit")) and str(r.get("created_at") or r.get("created_at_unit")).strip()]
updated_ats = [str(r.get("updated_at") or r.get("updated_at_unit")) for _, r in df_cm_merged.iterrows() if pd.notna(r.get("updated_at") or r.get("updated_at_unit")) and str(r.get("updated_at") or r.get("updated_at_unit")).strip()]
now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S.000")
oldest_created = min(created_ats) if created_ats else now_str
newest_updated = max(updated_ats) if updated_ats else now_str

barang_num = f"{SUB_CODE}-0001"
lot_num = f"LOT-0001-26-{barang_num}"
org_id = resolve_organizer_id(df_cm_merged.iloc[0].get("organizer"))

b_dict = {
    "id": 1,
    "number": barang_num,
    "subcategory_id": sub_map[SUB_CODE],
    "brand_id": "123",
    "uom_id": 1,
    "name": "Cermin Cembung (Convex)",
    "specification": "Convex 45cm",
    "min_stock_threshold": None,
    "image_url": first_img,
    "last_restock_at": None,
    "created_at": oldest_created,
    "updated_at": newest_updated
}

l_dict = {
    "id": 1,
    "number": lot_num,
    "barang_id": 1,
    "organizer_id": int(org_id),
    "vendor_id": None,
    "legacy_vendor_id": None,
    "location_id": None,
    "initial_quantity": None,
    "current_quantity": None,
    "po_number": None,
    "date_of_receipt": now_str,
    "unit_price": None,
    "image_url": first_img,
    "burden": None,
    "project_id": None,
    "created_at": now_str,
    "updated_at": now_str
}

df_sub_barangs = pd.DataFrame([b_dict])
df_sub_lots = pd.DataFrame([l_dict])

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f"{SUB_CODE}.csv", index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f"{SUB_CODE}.csv", index=False)

df_cm_units["lot_id"] = 1
for c in ["id", "lot_id", "location_id", "project_id", "vendor_id"]:
    if c in df_cm_units.columns:
        df_cm_units[c] = pd.to_numeric(df_cm_units[c], errors="coerce").astype("Int64")
df_cm_units.to_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

assert len(df_sub_barangs) == 1, f"Expected 1 barang, got {len(df_sub_barangs)}"
assert len(df_cm_units) == 7, f"Expected 7 units, got {len(df_cm_units)}"
print(f"\n[SUCCESS] Consolidated ACC-CM from 2 barangs down to 1 barang across 7 units.")

cm_display = df_sub_barangs[["id", "number", "brand_id", "name", "specification"]].copy()
brand_id_to_name = dict(zip(df_brand["id"], df_brand["name"]))
cm_display["brand"] = cm_display["brand_id"].astype(str).map(brand_id_to_name)
cm_display["unit_count"] = len(df_cm_units)
display(cm_display[["id", "number", "brand", "name", "specification", "unit_count"]])


Consolidating ACC-CM...
 Master Sync Complete: 1,443 barangs, 1,443 lots, 6,943 units.

[SUCCESS] Consolidated ACC-CM from 2 barangs down to 1 barang across 7 units.


,id,number,brand,name,specification,unit_count
0,1,ACC-CM-0001,No Brand,Cermin Cembung (Convex),Convex 45cm,7


---
## 4. Subcategory ACC-DP: Dispenser Consolidation
### Normalization Rules applied:
1. Combine ACC-DP-0001 and ACC-DP-0002 into one entry using ACC-DP-0001 data (, , Miyako).
2. Convert ACC-DP-0003 name to Title Case (, Denpoo).

In [5]:
# =====================================================================
# Subcategory ACC-DP: Dispenser Deduplication & Consolidation
# =====================================================================
SUB_CODE = "ACC-DP"
print(f"Consolidating {SUB_CODE}...")

df_dp_assets = pd.read_csv(STAGING_ASSETS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_dp_units = pd.read_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", dtype=str)

df_dp_merged = df_dp_assets.merge(
    df_dp_units[["ams_asset_id", "image_url", "created_at", "updated_at"]],
    left_on="id",
    right_on="ams_asset_id",
    how="left",
    suffixes=("", "_unit")
)

# Separate items: Group 1 (Miyako) and Group 2 (Denpoo)
# ams_asset_id 9419 is Denpoo (04565-ACC-DP-CFS-PTRE-24)
group1_rows = df_dp_merged[df_dp_merged["id"] != "9419"].copy()
group2_rows = df_dp_merged[df_dp_merged["id"] == "9419"].copy()

now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S.000")
sub_barangs = []
sub_lots = []
unit_lot_map = {}

# Group 1: Miyako Dispenser Otomatis
first_img_1 = None
for _, r in group1_rows.iterrows():
    img = r.get("image_url") or r.get("image_url_unit")
    if pd.notna(img) and str(img).strip():
        first_img_1 = str(img).strip()
        break

c_ats_1 = [str(r.get("created_at") or r.get("created_at_unit")) for _, r in group1_rows.iterrows() if pd.notna(r.get("created_at") or r.get("created_at_unit")) and str(r.get("created_at") or r.get("created_at_unit")).strip()]
u_ats_1 = [str(r.get("updated_at") or r.get("updated_at_unit")) for _, r in group1_rows.iterrows() if pd.notna(r.get("updated_at") or r.get("updated_at_unit")) and str(r.get("updated_at") or r.get("updated_at_unit")).strip()]

b1 = {
    "id": 1,
    "number": f"{SUB_CODE}-0001",
    "subcategory_id": sub_map[SUB_CODE],
    "brand_id": "101",
    "uom_id": 1,
    "name": "Dispenser",
    "specification": "Dispenser Otomatis",
    "min_stock_threshold": None,
    "image_url": first_img_1,
    "last_restock_at": None,
    "created_at": min(c_ats_1) if c_ats_1 else now_str,
    "updated_at": max(u_ats_1) if u_ats_1 else now_str
}
l1 = {
    "id": 1,
    "number": f"LOT-0001-26-{SUB_CODE}-0001",
    "barang_id": 1,
    "organizer_id": int(resolve_organizer_id(group1_rows.iloc[0].get("organizer"))),
    "vendor_id": None,
    "legacy_vendor_id": None,
    "location_id": None,
    "initial_quantity": None,
    "current_quantity": None,
    "po_number": None,
    "date_of_receipt": now_str,
    "unit_price": None,
    "image_url": first_img_1,
    "burden": None,
    "project_id": None,
    "created_at": now_str,
    "updated_at": now_str
}
sub_barangs.append(b1)
sub_lots.append(l1)
for _, r in group1_rows.iterrows():
    unit_lot_map[str(r["id"]).strip()] = 1

# Group 2: Denpoo Dispenser Kecil
first_img_2 = None
for _, r in group2_rows.iterrows():
    img = r.get("image_url") or r.get("image_url_unit")
    if pd.notna(img) and str(img).strip():
        first_img_2 = str(img).strip()
        break

c_ats_2 = [str(r.get("created_at") or r.get("created_at_unit")) for _, r in group2_rows.iterrows() if pd.notna(r.get("created_at") or r.get("created_at_unit")) and str(r.get("created_at") or r.get("created_at_unit")).strip()]
u_ats_2 = [str(r.get("updated_at") or r.get("updated_at_unit")) for _, r in group2_rows.iterrows() if pd.notna(r.get("updated_at") or r.get("updated_at_unit")) and str(r.get("updated_at") or r.get("updated_at_unit")).strip()]

b2 = {
    "id": 2,
    "number": f"{SUB_CODE}-0002",
    "subcategory_id": sub_map[SUB_CODE],
    "brand_id": "143",
    "uom_id": 1,
    "name": "Dispenser Kecil",
    "specification": None,
    "min_stock_threshold": None,
    "image_url": first_img_2,
    "last_restock_at": None,
    "created_at": min(c_ats_2) if c_ats_2 else now_str,
    "updated_at": max(u_ats_2) if u_ats_2 else now_str
}
l2 = {
    "id": 2,
    "number": f"LOT-0001-26-{SUB_CODE}-0002",
    "barang_id": 2,
    "organizer_id": int(resolve_organizer_id(group2_rows.iloc[0].get("organizer"))),
    "vendor_id": None,
    "legacy_vendor_id": None,
    "location_id": None,
    "initial_quantity": None,
    "current_quantity": None,
    "po_number": None,
    "date_of_receipt": now_str,
    "unit_price": None,
    "image_url": first_img_2,
    "burden": None,
    "project_id": None,
    "created_at": now_str,
    "updated_at": now_str
}
sub_barangs.append(b2)
sub_lots.append(l2)
for _, r in group2_rows.iterrows():
    unit_lot_map[str(r["id"]).strip()] = 2

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f"{SUB_CODE}.csv", index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f"{SUB_CODE}.csv", index=False)

for idx in range(len(df_dp_units)):
    ams_id = str(df_dp_units.at[idx, "ams_asset_id"]).strip()
    if ams_id in unit_lot_map:
        df_dp_units.at[idx, "lot_id"] = unit_lot_map[ams_id]

for c in ["id", "lot_id", "location_id", "project_id", "vendor_id"]:
    if c in df_dp_units.columns:
        df_dp_units[c] = pd.to_numeric(df_dp_units[c], errors="coerce").astype("Int64")
df_dp_units.to_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

assert len(df_sub_barangs) == 2, f"Expected 2 barangs, got {len(df_sub_barangs)}"
assert len(df_dp_units) == 9, f"Expected 9 units, got {len(df_dp_units)}"
print(f"\n[SUCCESS] Consolidated ACC-DP from 3 barangs down to 2 barangs across 9 units.")

dp_display = df_sub_barangs[["id", "number", "brand_id", "name", "specification"]].copy()
brand_id_to_name = dict(zip(df_brand["id"], df_brand["name"]))
dp_display["brand"] = dp_display["brand_id"].astype(str).map(brand_id_to_name)
dp_unit_counts = df_dp_units.groupby("lot_id").size().to_dict()
dp_display["unit_count"] = dp_display["id"].astype(int).map(dp_unit_counts)
display(dp_display[["id", "number", "brand", "name", "specification", "unit_count"]])


Consolidating ACC-DP...
 Master Sync Complete: 1,443 barangs, 1,443 lots, 6,943 units.

[SUCCESS] Consolidated ACC-DP from 3 barangs down to 2 barangs across 9 units.


,id,number,brand,name,specification,unit_count
0,1,ACC-DP-0001,Sanken,Dispenser,Dispenser Otomatis,8
1,2,ACC-DP-0002,Sonic,Dispenser Kecil,None,1


---
## 5. Subcategory ACC-GL: Glassboard Consolidation
### Normalization Rules applied:
1. **Standardize Typo**: Name normalized to  (Title Case).
2. **Specification Cleanup**: Strip non-size info (, ,  -> None) and embedded locations.
3. **Size Dimensions**: Canonicalized to  (Title Case).
4. **Grouping**: Group by .

In [6]:
# =====================================================================
# Subcategory ACC-GL: Glassboard Deduplication & Consolidation
# =====================================================================
SUB_CODE = "ACC-GL"
print(f"Consolidating {SUB_CODE}...")

def normalize_gl_spec(raw_spec):
    if not raw_spec or pd.isna(raw_spec):
        return None
    s = str(raw_spec).strip()
    m = re.search(r"(\d+\s*[xX]\s*\d+(?:\s*cm)?)", s)
    if m:
        dim = m.group(1).lower().replace("cm", "").strip()
        parts = re.split(r"\s*[xX]\s*", dim)
        if len(parts) == 2:
            return f"{parts[0].strip()} x {parts[1].strip()} cm"
        return m.group(1).strip()
    return None

df_gl_assets = pd.read_csv(STAGING_ASSETS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_gl_units = pd.read_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", dtype=str)

df_gl_merged = df_gl_assets.merge(
    df_gl_units[["ams_asset_id", "image_url", "created_at", "updated_at"]],
    left_on="id",
    right_on="ams_asset_id",
    how="left",
    suffixes=("", "_unit")
)

groups = defaultdict(list)
for _, r in df_gl_merged.iterrows():
    b_id = resolve_brand_id(r.get("brand"))
    spec_norm = normalize_gl_spec(r.get("spesifikasi"))
    spec_key = spec_norm if spec_norm else ""
    groups[(b_id, spec_key)].append(r)

sorted_groups = sorted(groups.items(), key=lambda item: int(float(item[1][0]["id"])))

sub_barangs = []
sub_lots = []
unit_lot_map = {}
seq_counter = 1
now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S.000")

for (b_id, spec_key), group_rows in sorted_groups:
    barang_num = f"{SUB_CODE}-{seq_counter:04d}"
    lot_num = f"LOT-0001-26-{barang_num}"
    display_spec = spec_key if spec_key else None
    
    first_img = None
    for r in group_rows:
        img = r.get("image_url") or r.get("image_url_unit")
        if pd.notna(img) and str(img).strip():
            first_img = str(img).strip()
            break
            
    created_ats = [str(r.get("created_at") or r.get("created_at_unit")) for r in group_rows if pd.notna(r.get("created_at") or r.get("created_at_unit")) and str(r.get("created_at") or r.get("created_at_unit")).strip()]
    updated_ats = [str(r.get("updated_at") or r.get("updated_at_unit")) for r in group_rows if pd.notna(r.get("updated_at") or r.get("updated_at_unit")) and str(r.get("updated_at") or r.get("updated_at_unit")).strip()]
    
    oldest_created = min(created_ats) if created_ats else now_str
    newest_updated = max(updated_ats) if updated_ats else now_str
    org_id = resolve_organizer_id(group_rows[0].get("organizer"))
    
    b_dict = {
        "id": seq_counter,
        "number": barang_num,
        "subcategory_id": sub_map[SUB_CODE],
        "brand_id": b_id,
        "uom_id": 1,
        "name": "Glassboard",
        "specification": display_spec,
        "min_stock_threshold": None,
        "image_url": first_img,
        "last_restock_at": None,
        "created_at": oldest_created,
        "updated_at": newest_updated
    }
    
    l_dict = {
        "id": seq_counter,
        "number": lot_num,
        "barang_id": seq_counter,
        "organizer_id": int(org_id),
        "vendor_id": None,
        "legacy_vendor_id": None,
        "location_id": None,
        "initial_quantity": None,
        "current_quantity": None,
        "po_number": None,
        "date_of_receipt": now_str,
        "unit_price": None,
        "image_url": first_img,
        "burden": None,
        "project_id": None,
        "created_at": now_str,
        "updated_at": now_str
    }
    
    sub_barangs.append(b_dict)
    sub_lots.append(l_dict)
    
    for r in group_rows:
        unit_lot_map[str(r["id"]).strip()] = seq_counter
        
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)

df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f"{SUB_CODE}.csv", index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f"{SUB_CODE}.csv", index=False)

for idx in range(len(df_gl_units)):
    ams_id = str(df_gl_units.at[idx, "ams_asset_id"]).strip()
    if ams_id in unit_lot_map:
        df_gl_units.at[idx, "lot_id"] = unit_lot_map[ams_id]

for c in ["id", "lot_id", "location_id", "project_id", "vendor_id"]:
    if c in df_gl_units.columns:
        df_gl_units[c] = pd.to_numeric(df_gl_units[c], errors="coerce").astype("Int64")
df_gl_units.to_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()

assert len(df_sub_barangs) == 7, f"Expected 7 barangs, got {len(df_sub_barangs)}"
assert len(df_gl_units) == 54, f"Expected 54 units, got {len(df_gl_units)}"
print(f"\n[SUCCESS] Consolidated ACC-GL from 12 barangs down to 7 barangs across 54 units.")

gl_display = df_sub_barangs[["id", "number", "brand_id", "name", "specification"]].copy()
brand_id_to_name = dict(zip(df_brand["id"], df_brand["name"]))
gl_display["brand"] = gl_display["brand_id"].astype(str).map(brand_id_to_name)
gl_unit_counts = df_gl_units.groupby("lot_id").size().to_dict()
gl_display["unit_count"] = gl_display["id"].astype(int).map(gl_unit_counts)
display(gl_display[["id", "number", "brand", "name", "specification", "unit_count"]])


Consolidating ACC-GL...
 Master Sync Complete: 1,443 barangs, 1,443 lots, 6,943 units.

[SUCCESS] Consolidated ACC-GL from 12 barangs down to 7 barangs across 54 units.


,id,number,brand,name,specification,unit_count
0,1,ACC-GL-0001,Syly Art,Glassboard,90 x 80 cm,2
1,2,ACC-GL-0002,Syly Art,Glassboard,None,46
2,3,ACC-GL-0003,Syly Art,Glassboard,90 x 120 cm,1
3,4,ACC-GL-0004,Daya Kurnia,Glassboard,90 x 120 cm,2
4,5,ACC-GL-0005,Daya Kurnia,Glassboard,90 x 140 cm,1
5,6,ACC-GL-0006,Daya Kurnia,Glassboard,90 x 90 cm,1
6,7,ACC-GL-0007,No Brand,Glassboard,None,1


---
## 6. Formatting Polish: ACC-FT & ACC-GE (Title Case)
### Formatting Rules applied:
1. **ACC-FT**: Standardize name  -> .
2. **ACC-GE**: Standardize name  -> .

In [7]:
# =====================================================================
# Formatting Polish: ACC-FT & ACC-GE (Title Case Names)
# =====================================================================
# 1. Update ACC-FT
df_ft = pd.read_csv(STAGING_BARANGS_DIR / "ACC-FT.csv", dtype=str)
df_ft["name"] = df_ft["name"].apply(lambda x: str(x).strip().title() if pd.notna(x) else x)
df_ft.to_csv(STAGING_BARANGS_DIR / "ACC-FT.csv", index=False)
print("ACC-FT names formatted to Title Case:")
display(df_ft[["number", "name"]])

# 2. Update ACC-GE
df_ge = pd.read_csv(STAGING_BARANGS_DIR / "ACC-GE.csv", dtype=str)
df_ge["name"] = df_ge["name"].apply(lambda x: str(x).strip().title() if pd.notna(x) else x)
df_ge.to_csv(STAGING_BARANGS_DIR / "ACC-GE.csv", index=False)
print("ACC-GE names formatted to Title Case:")
display(df_ge[["number", "name"]])

# Synchronize global deliverables
df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()
print(f"\n[SUCCESS] Formatted ACC-FT & ACC-GE to Title Case and synchronized masters.")


ACC-FT names formatted to Title Case:


,number,name
0,ACC-FT-0001,Foto


ACC-GE names formatted to Title Case:


,number,name
0,ACC-GE-0001,Gerobak Etalase
1,ACC-GE-0002,Gerobak Etalase Kantin


 Master Sync Complete: 1,443 barangs, 1,443 lots, 6,943 units.

[SUCCESS] Formatted ACC-FT & ACC-GE to Title Case and synchronized masters.


---
## 7. Subcategory ACC-P3K: Kotak P3K Consolidation
### Normalization Rules applied:
1. Rename  ->  in specification.
2. Merge typo entry  (Kotoak P3K) into  (Kotak P3K).

In [8]:
# =====================================================================
# Subcategory ACC-P3K: Kotak P3K Deduplication & Consolidation
# =====================================================================
SUB_CODE = "ACC-P3K"
print(f"Consolidating {SUB_CODE}...")

df_assets = pd.read_csv(STAGING_ASSETS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_units = pd.read_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_merged = df_assets.merge(
    df_units[["ams_asset_id", "image_url", "created_at", "updated_at"]],
    left_on="id", right_on="ams_asset_id", how="left", suffixes=("", "_unit")
)

def norm_p3k_spec(s):
    if not s or pd.isna(s): return None
    val = str(s).strip()
    if val.upper() in ["STENLIST", "STENLIS", "STAINLESS"]:
        return "Stainless"
    return val

groups = defaultdict(list)
for _, r in df_merged.iterrows():
    b_id = resolve_brand_id(r.get("brand"))
    spec = norm_p3k_spec(r.get("spesifikasi"))
    spec_key = spec.upper() if spec else ""
    groups[(b_id, "Kotak P3K", spec_key)].append(r)

sorted_groups = sorted(groups.items(), key=lambda x: int(float(x[1][0]["id"])))
sub_barangs, sub_lots, unit_lot_map = [], [], {}
seq_counter = 1
now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S.000")

for (b_id, name_val, spec_key), group_rows in sorted_groups:
    barang_num = f"{SUB_CODE}-{seq_counter:04d}"
    lot_num = f"LOT-0001-26-{barang_num}"
    display_spec = norm_p3k_spec(group_rows[0].get("spesifikasi"))
    first_img = next((str(r.get("image_url") or r.get("image_url_unit")).strip() for _, r in enumerate(group_rows) if pd.notna(r.get("image_url") or r.get("image_url_unit")) and str(r.get("image_url") or r.get("image_url_unit")).strip()), None)
    c_ats = [str(r.get("created_at") or r.get("created_at_unit")) for r in group_rows if pd.notna(r.get("created_at") or r.get("created_at_unit")) and str(r.get("created_at") or r.get("created_at_unit")).strip()]
    u_ats = [str(r.get("updated_at") or r.get("updated_at_unit")) for r in group_rows if pd.notna(r.get("updated_at") or r.get("updated_at_unit")) and str(r.get("updated_at") or r.get("updated_at_unit")).strip()]
    org_id = resolve_organizer_id(group_rows[0].get("organizer"))

    sub_barangs.append({
        "id": seq_counter, "number": barang_num, "subcategory_id": sub_map[SUB_CODE],
        "brand_id": b_id, "uom_id": 1, "name": name_val, "specification": display_spec,
        "min_stock_threshold": None, "image_url": first_img, "last_restock_at": None,
        "created_at": min(c_ats) if c_ats else now_str, "updated_at": max(u_ats) if u_ats else now_str
    })
    sub_lots.append({
        "id": seq_counter, "number": lot_num, "barang_id": seq_counter, "organizer_id": int(org_id),
        "vendor_id": None, "legacy_vendor_id": None, "location_id": None, "initial_quantity": None,
        "current_quantity": None, "po_number": None, "date_of_receipt": now_str, "unit_price": None,
        "image_url": first_img, "burden": None, "project_id": None, "created_at": now_str, "updated_at": now_str
    })
    for r in group_rows: unit_lot_map[str(r["id"]).strip()] = seq_counter
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)
df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f"{SUB_CODE}.csv", index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f"{SUB_CODE}.csv", index=False)
for idx in range(len(df_units)):
    ams_id = str(df_units.at[idx, "ams_asset_id"]).strip()
    if ams_id in unit_lot_map: df_units.at[idx, "lot_id"] = unit_lot_map[ams_id]
for c in ["id", "lot_id", "location_id", "project_id", "vendor_id"]:
    if c in df_units.columns: df_units[c] = pd.to_numeric(df_units[c], errors="coerce").astype("Int64")
df_units.to_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", index=False)
df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()
assert len(df_sub_barangs) == 4, f"Expected 4 barangs, got {len(df_sub_barangs)}"
print(f"\n[SUCCESS] Consolidated ACC-P3K from 5 barangs down to 4 barangs.")
display(df_sub_barangs[["id", "number", "brand_id", "name", "specification"]])


Consolidating ACC-P3K...
 Master Sync Complete: 1,443 barangs, 1,443 lots, 6,943 units.

[SUCCESS] Consolidated ACC-P3K from 5 barangs down to 4 barangs.


,id,number,brand_id,name,specification
0,1,ACC-P3K-0001,64,Kotak P3K,Stainless
1,2,ACC-P3K-0002,138,Kotak P3K,None
2,3,ACC-P3K-0003,64,Kotak P3K,Tinggi 45 x Lebar 30 Tebal 12 cm
3,4,ACC-P3K-0004,64,Kotak P3K,None


---
## 8. Subcategory ACC-PH: Pengharum Ruangan Consolidation
### Normalization Rules applied:
1. Strip brand names (Glade, Stella) from specifications.
2. Canonicalize name to  (standardizes ).
3. Combine 0006, 0007, 0010 (Stella, spec None) and 0005, 0008, 0009 (Glade, spec None).

In [9]:
# =====================================================================
# Subcategory ACC-PH: Pengharum Ruangan Deduplication & Consolidation
# =====================================================================
SUB_CODE = "ACC-PH"
print(f"Consolidating {SUB_CODE}...")

df_assets = pd.read_csv(STAGING_ASSETS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_units = pd.read_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_merged = df_assets.merge(
    df_units[["ams_asset_id", "image_url", "created_at", "updated_at"]],
    left_on="id", right_on="ams_asset_id", how="left", suffixes=("", "_unit")
)

def norm_ph_spec(s):
    if not s or pd.isna(s): return None
    val = str(s).strip()
    if val.upper() in ["GLADE", "STELLA", "NULL", "NONE", "NAN"]:
        return None
    return val

groups = defaultdict(list)
for _, r in df_merged.iterrows():
    b_id = resolve_brand_id(r.get("brand"))
    spec = norm_ph_spec(r.get("spesifikasi"))
    spec_key = spec.upper() if spec else ""
    groups[(b_id, "Pengharum Ruangan", spec_key)].append(r)

sorted_groups = sorted(groups.items(), key=lambda x: int(float(x[1][0]["id"])))
sub_barangs, sub_lots, unit_lot_map = [], [], {}
seq_counter = 1
now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S.000")

for (b_id, name_val, spec_key), group_rows in sorted_groups:
    barang_num = f"{SUB_CODE}-{seq_counter:04d}"
    lot_num = f"LOT-0001-26-{barang_num}"
    display_spec = norm_ph_spec(group_rows[0].get("spesifikasi"))
    first_img = next((str(r.get("image_url") or r.get("image_url_unit")).strip() for _, r in enumerate(group_rows) if pd.notna(r.get("image_url") or r.get("image_url_unit")) and str(r.get("image_url") or r.get("image_url_unit")).strip()), None)
    c_ats = [str(r.get("created_at") or r.get("created_at_unit")) for r in group_rows if pd.notna(r.get("created_at") or r.get("created_at_unit")) and str(r.get("created_at") or r.get("created_at_unit")).strip()]
    u_ats = [str(r.get("updated_at") or r.get("updated_at_unit")) for r in group_rows if pd.notna(r.get("updated_at") or r.get("updated_at_unit")) and str(r.get("updated_at") or r.get("updated_at_unit")).strip()]
    org_id = resolve_organizer_id(group_rows[0].get("organizer"))

    sub_barangs.append({
        "id": seq_counter, "number": barang_num, "subcategory_id": sub_map[SUB_CODE],
        "brand_id": b_id, "uom_id": 1, "name": name_val, "specification": display_spec,
        "min_stock_threshold": None, "image_url": first_img, "last_restock_at": None,
        "created_at": min(c_ats) if c_ats else now_str, "updated_at": max(u_ats) if u_ats else now_str
    })
    sub_lots.append({
        "id": seq_counter, "number": lot_num, "barang_id": seq_counter, "organizer_id": int(org_id),
        "vendor_id": None, "legacy_vendor_id": None, "location_id": None, "initial_quantity": None,
        "current_quantity": None, "po_number": None, "date_of_receipt": now_str, "unit_price": None,
        "image_url": first_img, "burden": None, "project_id": None, "created_at": now_str, "updated_at": now_str
    })
    for r in group_rows: unit_lot_map[str(r["id"]).strip()] = seq_counter
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)
df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f"{SUB_CODE}.csv", index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f"{SUB_CODE}.csv", index=False)
for idx in range(len(df_units)):
    ams_id = str(df_units.at[idx, "ams_asset_id"]).strip()
    if ams_id in unit_lot_map: df_units.at[idx, "lot_id"] = unit_lot_map[ams_id]
for c in ["id", "lot_id", "location_id", "project_id", "vendor_id"]:
    if c in df_units.columns: df_units[c] = pd.to_numeric(df_units[c], errors="coerce").astype("Int64")
df_units.to_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", index=False)
df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()
assert len(df_sub_barangs) == 6, f"Expected 6 barangs, got {len(df_sub_barangs)}"
print(f"\n[SUCCESS] Consolidated ACC-PH from 10 barangs down to 6 barangs.")
display(df_sub_barangs[["id", "number", "brand_id", "name", "specification"]])


Consolidating ACC-PH...
 Master Sync Complete: 1,443 barangs, 1,443 lots, 6,943 units.

[SUCCESS] Consolidated ACC-PH from 10 barangs down to 6 barangs.


,id,number,brand_id,name,specification
0,1,ACC-PH-0001,120,Pengharum Ruangan,Pengharum Otomatis
1,2,ACC-PH-0002,175,Pengharum Ruangan,Warna Putih
2,3,ACC-PH-0003,120,Pengharum Ruangan,Pengarum Tempel
3,4,ACC-PH-0004,120,Pengharum Ruangan,Warna Putih
4,5,ACC-PH-0005,120,Pengharum Ruangan,None
5,6,ACC-PH-0006,175,Pengharum Ruangan,None


---
## 9. Subcategory ACC-PM: Perlengkapan Motor Consolidation
### Normalization Rules applied:
1. Strip personal names e.g. , .
2. Title Case names and specs.
3. Combine Helm into  (JS, ID 171) and Jas Hujan into  (Tiger Head, ID 172).

In [10]:
# =====================================================================
# Subcategory ACC-PM: Perlengkapan Motor Deduplication & Consolidation
# =====================================================================
SUB_CODE = "ACC-PM"
print(f"Consolidating {SUB_CODE}...")

df_assets = pd.read_csv(STAGING_ASSETS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_units = pd.read_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_merged = df_assets.merge(
    df_units[["ams_asset_id", "image_url", "created_at", "updated_at"]],
    left_on="id", right_on="ams_asset_id", how="left", suffixes=("", "_unit")
)

groups = defaultdict(list)
for _, r in df_merged.iterrows():
    n = str(r.get("name")).strip().upper()
    if "HELM" in n:
        groups[("171", "Helm Motor Op", "")].append(r)
    else:
        groups[("172", "Jas Hujan Motor Op", "")].append(r)

sorted_groups = sorted(groups.items(), key=lambda x: int(float(x[1][0]["id"])))
sub_barangs, sub_lots, unit_lot_map = [], [], {}
seq_counter = 1
now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S.000")

for (b_id, name_val, spec_key), group_rows in sorted_groups:
    barang_num = f"{SUB_CODE}-{seq_counter:04d}"
    lot_num = f"LOT-0001-26-{barang_num}"
    first_img = next((str(r.get("image_url") or r.get("image_url_unit")).strip() for _, r in enumerate(group_rows) if pd.notna(r.get("image_url") or r.get("image_url_unit")) and str(r.get("image_url") or r.get("image_url_unit")).strip()), None)
    c_ats = [str(r.get("created_at") or r.get("created_at_unit")) for r in group_rows if pd.notna(r.get("created_at") or r.get("created_at_unit")) and str(r.get("created_at") or r.get("created_at_unit")).strip()]
    u_ats = [str(r.get("updated_at") or r.get("updated_at_unit")) for r in group_rows if pd.notna(r.get("updated_at") or r.get("updated_at_unit")) and str(r.get("updated_at") or r.get("updated_at_unit")).strip()]
    org_id = resolve_organizer_id(group_rows[0].get("organizer"))

    sub_barangs.append({
        "id": seq_counter, "number": barang_num, "subcategory_id": sub_map[SUB_CODE],
        "brand_id": b_id, "uom_id": 1, "name": name_val, "specification": None,
        "min_stock_threshold": None, "image_url": first_img, "last_restock_at": None,
        "created_at": min(c_ats) if c_ats else now_str, "updated_at": max(u_ats) if u_ats else now_str
    })
    sub_lots.append({
        "id": seq_counter, "number": lot_num, "barang_id": seq_counter, "organizer_id": int(org_id),
        "vendor_id": None, "legacy_vendor_id": None, "location_id": None, "initial_quantity": None,
        "current_quantity": None, "po_number": None, "date_of_receipt": now_str, "unit_price": None,
        "image_url": first_img, "burden": None, "project_id": None, "created_at": now_str, "updated_at": now_str
    })
    for r in group_rows: unit_lot_map[str(r["id"]).strip()] = seq_counter
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)
df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f"{SUB_CODE}.csv", index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f"{SUB_CODE}.csv", index=False)
for idx in range(len(df_units)):
    ams_id = str(df_units.at[idx, "ams_asset_id"]).strip()
    if ams_id in unit_lot_map: df_units.at[idx, "lot_id"] = unit_lot_map[ams_id]
for c in ["id", "lot_id", "location_id", "project_id", "vendor_id"]:
    if c in df_units.columns: df_units[c] = pd.to_numeric(df_units[c], errors="coerce").astype("Int64")
df_units.to_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", index=False)
df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()
assert len(df_sub_barangs) == 2, f"Expected 2 barangs, got {len(df_sub_barangs)}"
print(f"\n[SUCCESS] Consolidated ACC-PM from 5 barangs down to 2 barangs.")
display(df_sub_barangs[["id", "number", "brand_id", "name", "specification"]])


Consolidating ACC-PM...
 Master Sync Complete: 1,443 barangs, 1,443 lots, 6,943 units.

[SUCCESS] Consolidated ACC-PM from 5 barangs down to 2 barangs.


,id,number,brand_id,name,specification
0,1,ACC-PM-0001,171,Helm Motor Op,None
1,2,ACC-PM-0002,172,Jas Hujan Motor Op,None


---
## 10. Subcategory ACC-POT: Pot Tanaman Consolidation
### Normalization Rules applied:
1. Standardize names and specs to Title Case.
2. Combine  (POT) into  (Pot Tanaman, spec None).

In [11]:
# =====================================================================
# Subcategory ACC-POT: Pot Tanaman Deduplication & Consolidation
# =====================================================================
SUB_CODE = "ACC-POT"
print(f"Consolidating {SUB_CODE}...")

df_assets = pd.read_csv(STAGING_ASSETS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_units = pd.read_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_merged = df_assets.merge(
    df_units[["ams_asset_id", "image_url", "created_at", "updated_at"]],
    left_on="id", right_on="ams_asset_id", how="left", suffixes=("", "_unit")
)

def norm_pot_spec(s):
    if not s or pd.isna(s) or str(s).strip().upper() in ["NAN", "NULL", "NONE"]: return None
    val = str(s).strip()
    if "TANAH LIAT" in val.upper(): return "Tanah Liat Coklat"
    if "WARNA COKLAT" in val.upper(): return "Warna Coklat"
    if "MATI" in val.upper(): return "Tanaman Mati"
    return val.title()

groups = defaultdict(list)
for _, r in df_merged.iterrows():
    b_id = resolve_brand_id(r.get("brand"))
    spec = norm_pot_spec(r.get("spesifikasi"))
    spec_key = spec.upper() if spec else ""
    groups[(b_id, "Pot Tanaman", spec_key)].append(r)

sorted_groups = sorted(groups.items(), key=lambda x: int(float(x[1][0]["id"])))
sub_barangs, sub_lots, unit_lot_map = [], [], {}
seq_counter = 1
now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S.000")

for (b_id, name_val, spec_key), group_rows in sorted_groups:
    barang_num = f"{SUB_CODE}-{seq_counter:04d}"
    lot_num = f"LOT-0001-26-{barang_num}"
    display_spec = norm_pot_spec(group_rows[0].get("spesifikasi"))
    first_img = next((str(r.get("image_url") or r.get("image_url_unit")).strip() for _, r in enumerate(group_rows) if pd.notna(r.get("image_url") or r.get("image_url_unit")) and str(r.get("image_url") or r.get("image_url_unit")).strip()), None)
    c_ats = [str(r.get("created_at") or r.get("created_at_unit")) for r in group_rows if pd.notna(r.get("created_at") or r.get("created_at_unit")) and str(r.get("created_at") or r.get("created_at_unit")).strip()]
    u_ats = [str(r.get("updated_at") or r.get("updated_at_unit")) for r in group_rows if pd.notna(r.get("updated_at") or r.get("updated_at_unit")) and str(r.get("updated_at") or r.get("updated_at_unit")).strip()]
    org_id = resolve_organizer_id(group_rows[0].get("organizer"))

    sub_barangs.append({
        "id": seq_counter, "number": barang_num, "subcategory_id": sub_map[SUB_CODE],
        "brand_id": b_id, "uom_id": 1, "name": name_val, "specification": display_spec,
        "min_stock_threshold": None, "image_url": first_img, "last_restock_at": None,
        "created_at": min(c_ats) if c_ats else now_str, "updated_at": max(u_ats) if u_ats else now_str
    })
    sub_lots.append({
        "id": seq_counter, "number": lot_num, "barang_id": seq_counter, "organizer_id": int(org_id),
        "vendor_id": None, "legacy_vendor_id": None, "location_id": None, "initial_quantity": None,
        "current_quantity": None, "po_number": None, "date_of_receipt": now_str, "unit_price": None,
        "image_url": first_img, "burden": None, "project_id": None, "created_at": now_str, "updated_at": now_str
    })
    for r in group_rows: unit_lot_map[str(r["id"]).strip()] = seq_counter
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)
df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f"{SUB_CODE}.csv", index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f"{SUB_CODE}.csv", index=False)
for idx in range(len(df_units)):
    ams_id = str(df_units.at[idx, "ams_asset_id"]).strip()
    if ams_id in unit_lot_map: df_units.at[idx, "lot_id"] = unit_lot_map[ams_id]
for c in ["id", "lot_id", "location_id", "project_id", "vendor_id"]:
    if c in df_units.columns: df_units[c] = pd.to_numeric(df_units[c], errors="coerce").astype("Int64")
df_units.to_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", index=False)
df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()
assert len(df_sub_barangs) == 4, f"Expected 4 barangs, got {len(df_sub_barangs)}"
print(f"\n[SUCCESS] Consolidated ACC-POT from 5 barangs down to 4 barangs.")
display(df_sub_barangs[["id", "number", "brand_id", "name", "specification"]])


Consolidating ACC-POT...
 Master Sync Complete: 1,443 barangs, 1,443 lots, 6,943 units.

[SUCCESS] Consolidated ACC-POT from 5 barangs down to 4 barangs.


,id,number,brand_id,name,specification
0,1,ACC-POT-0001,123,Pot Tanaman,Warna Coklat
1,2,ACC-POT-0002,123,Pot Tanaman,None
2,3,ACC-POT-0003,123,Pot Tanaman,Tanah Liat Coklat
3,4,ACC-POT-0004,123,Pot Tanaman,Tanaman Mati


---
## 11. Subcategories ACC-TAP & ACC-TK Consolidation
### Normalization Rules applied:
1. **ACC-TAP**: Rename  -> .
2. **ACC-TK**: Rename  -> , and fix  ->  in specs.

In [12]:
# =====================================================================
# Subcategories ACC-TAP & ACC-TK Renaming & Polish
# =====================================================================
# 1. ACC-TAP
df_tap = pd.read_csv(STAGING_BARANGS_DIR / "ACC-TAP.csv", dtype=str)
df_tap["name"] = df_tap["name"].replace({"Termos Stenlis": "Termos Stainless"})
df_tap.to_csv(STAGING_BARANGS_DIR / "ACC-TAP.csv", index=False)

# 2. ACC-TK
df_tk = pd.read_csv(STAGING_BARANGS_DIR / "ACC-TK.csv", dtype=str)
df_tk["name"] = df_tk["name"].replace({"Troly Stenlis Pantry": "Troli Stainless Pantry"})
df_tk["specification"] = df_tk["specification"].replace({
    "Stenlis": "Stainless",
    "Troli Stenlis Lt 1": "Troli Stainless Lt 1"
})
df_tk.to_csv(STAGING_BARANGS_DIR / "ACC-TK.csv", index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()
print("\n[SUCCESS] Updated ACC-TAP and ACC-TK.")
display(df_tap[["number", "name", "specification"]])
display(df_tk[["number", "name", "specification"]])


 Master Sync Complete: 1,443 barangs, 1,443 lots, 6,943 units.

[SUCCESS] Updated ACC-TAP and ACC-TK.


,number,name,specification
0,ACC-TAP-0001,Termos Stainless,5 Liter
1,ACC-TAP-0002,Termos Air Panas,NaN
2,ACC-TAP-0003,Termos Air Panas,NaN
3,ACC-TAP-0004,Termos Air Panas,NaN


,number,name,specification
0,ACC-TK-0001,Troli Stainless Pantry,Stainless
1,ACC-TK-0002,Troli Kecil,NaN
2,ACC-TK-0003,Troli Kecil,Troli Stainless Lt 1


---
## 12. Subcategory ACC-TS: Tempat Sampah Consolidation
### Normalization Rules applied:
1. Deprecate  -> canonical .
2. Strip  (and typos) and  from name; move  from name to specification.
3. Standardize stainless spec to .
4. Maintain distinct sizes (, , ).

In [13]:
# =====================================================================
# Subcategory ACC-TS: Tempat Sampah Deduplication & Consolidation
# =====================================================================
SUB_CODE = "ACC-TS"
print(f"Consolidating {SUB_CODE}...")

df_assets = pd.read_csv(STAGING_ASSETS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_units = pd.read_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_merged = df_assets.merge(
    df_units[["ams_asset_id", "image_url", "created_at", "updated_at"]],
    left_on="id", right_on="ams_asset_id", how="left", suffixes=("", "_unit")
)

def clean_ts_entry(raw_name, raw_spec):
    n = str(raw_name).strip() if pd.notna(raw_name) else "Tempat Sampah"
    s = str(raw_spec).strip() if pd.notna(raw_spec) else ""
    has_besi = "besi" in n.lower()
    n = re.sub(r"(?i)\btong\s+sampah\b", "Tempat Sampah", n)
    n = re.sub(r"(?i)\b(stenlis|stenlist|stanless|stanlees|srenlist|stainless|plastik|besi)\b", "", n)
    n = re.sub(r"\s+", " ", n).strip()
    if not n: n = "Tempat Sampah"
    n = n.title()
    
    is_stainless = bool(re.search(r"(?i)\b(stenlis|stenlist|stanless|stanlees|srenlist|stainless)\b", s))
    if is_stainless:
        s_clean = "Stainless"
    elif has_besi or "besi" in s.lower():
        s_clean = "Besi"
    elif "bundar" in s.lower():
        s_clean = "Bundar"
    elif "warna merah" in s.lower():
        s_clean = "Warna Merah"
    elif "ts-doble" in s.lower():
        s_clean = "TS-Doble"
    elif "12 liter" in s.lower():
        s_clean = "Plastik Tutup 12 Liter"
    elif "10 liter" in s.lower():
        s_clean = "Plastik Tutup 10 Liter"
    elif "60x30" in s.lower():
        s_clean = "Ukuran 60 x 30 cm"
    elif s == "" or s.lower() in ["nan", "null", "none"]:
        s_clean = None
    else:
        s_clean = s.title()
    return n, s_clean

groups = defaultdict(list)
for _, r in df_merged.iterrows():
    b_id = resolve_brand_id(r.get("brand"))
    c_name, c_spec = clean_ts_entry(r.get("name"), r.get("spesifikasi"))
    spec_key = c_spec.upper() if c_spec else ""
    groups[(b_id, c_name, spec_key)].append(r)

sorted_groups = sorted(groups.items(), key=lambda x: int(float(x[1][0]["id"])))
sub_barangs, sub_lots, unit_lot_map = [], [], {}
seq_counter = 1
now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S.000")

for (b_id, name_val, spec_key), group_rows in sorted_groups:
    barang_num = f"{SUB_CODE}-{seq_counter:04d}"
    lot_num = f"LOT-0001-26-{barang_num}"
    _, display_spec = clean_ts_entry(group_rows[0].get("name"), group_rows[0].get("spesifikasi"))
    first_img = next((str(r.get("image_url") or r.get("image_url_unit")).strip() for _, r in enumerate(group_rows) if pd.notna(r.get("image_url") or r.get("image_url_unit")) and str(r.get("image_url") or r.get("image_url_unit")).strip()), None)
    c_ats = [str(r.get("created_at") or r.get("created_at_unit")) for r in group_rows if pd.notna(r.get("created_at") or r.get("created_at_unit")) and str(r.get("created_at") or r.get("created_at_unit")).strip()]
    u_ats = [str(r.get("updated_at") or r.get("updated_at_unit")) for r in group_rows if pd.notna(r.get("updated_at") or r.get("updated_at_unit")) and str(r.get("updated_at") or r.get("updated_at_unit")).strip()]
    org_id = resolve_organizer_id(group_rows[0].get("organizer"))

    sub_barangs.append({
        "id": seq_counter, "number": barang_num, "subcategory_id": sub_map[SUB_CODE],
        "brand_id": b_id, "uom_id": 1, "name": name_val, "specification": display_spec,
        "min_stock_threshold": None, "image_url": first_img, "last_restock_at": None,
        "created_at": min(c_ats) if c_ats else now_str, "updated_at": max(u_ats) if u_ats else now_str
    })
    sub_lots.append({
        "id": seq_counter, "number": lot_num, "barang_id": seq_counter, "organizer_id": int(org_id),
        "vendor_id": None, "legacy_vendor_id": None, "location_id": None, "initial_quantity": None,
        "current_quantity": None, "po_number": None, "date_of_receipt": now_str, "unit_price": None,
        "image_url": first_img, "burden": None, "project_id": None, "created_at": now_str, "updated_at": now_str
    })
    for r in group_rows: unit_lot_map[str(r["id"]).strip()] = seq_counter
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)
df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f"{SUB_CODE}.csv", index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f"{SUB_CODE}.csv", index=False)
for idx in range(len(df_units)):
    ams_id = str(df_units.at[idx, "ams_asset_id"]).strip()
    if ams_id in unit_lot_map: df_units.at[idx, "lot_id"] = unit_lot_map[ams_id]
for c in ["id", "lot_id", "location_id", "project_id", "vendor_id"]:
    if c in df_units.columns: df_units[c] = pd.to_numeric(df_units[c], errors="coerce").astype("Int64")
df_units.to_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", index=False)
df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()
assert len(df_sub_barangs) == 17, f"Expected 17 barangs, got {len(df_sub_barangs)}"
print(f"\n[SUCCESS] Consolidated ACC-TS from 25 barangs down to 17 barangs.")
display(df_sub_barangs[["id", "number", "brand_id", "name", "specification"]])


Consolidating ACC-TS...
 Master Sync Complete: 1,443 barangs, 1,443 lots, 6,943 units.

[SUCCESS] Consolidated ACC-TS from 25 barangs down to 17 barangs.


,id,number,brand_id,name,specification
0,1,ACC-TS-0001,125,Tempat Sampah,Bundar
1,2,ACC-TS-0002,79,Tempat Sampah,Stainless
2,3,ACC-TS-0003,125,Tempat Sampah,Stainless
3,4,ACC-TS-0004,79,Tempat Sampah,None
4,5,ACC-TS-0005,64,Tempat Sampah,None
5,6,ACC-TS-0006,79,Tempat Sampah,Bundar
6,7,ACC-TS-0007,64,Tempat Sampah,TS-Doble
7,8,ACC-TS-0008,79,Tempat Sampah B3,Warna Merah
8,9,ACC-TS-0009,64,Tempat Sampah,Plastik Tutup 12 Liter
9,10,ACC-TS-0010,64,Tempat Sampah,Plastik Tutup 10 Liter


---
## 13. Subcategory ACC-WB: Whiteboard Consolidation
### Normalization Rules applied:
1. Combine  (White Board) and  (Whiteboard) into canonical  (No Brand).

In [14]:
# =====================================================================
# Subcategory ACC-WB: Whiteboard Deduplication & Consolidation
# =====================================================================
SUB_CODE = "ACC-WB"
print(f"Consolidating {SUB_CODE}...")

df_assets = pd.read_csv(STAGING_ASSETS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_units = pd.read_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", dtype=str)
df_merged = df_assets.merge(
    df_units[["ams_asset_id", "image_url", "created_at", "updated_at"]],
    left_on="id", right_on="ams_asset_id", how="left", suffixes=("", "_unit")
)

groups = defaultdict(list)
for _, r in df_merged.iterrows():
    b_id = resolve_brand_id(r.get("brand"))
    n = "Panaboard" if b_id == "25" else "Whiteboard"
    groups[(b_id, n, "")].append(r)

sorted_groups = sorted(groups.items(), key=lambda x: int(float(x[1][0]["id"])))
sub_barangs, sub_lots, unit_lot_map = [], [], {}
seq_counter = 1
now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S.000")

for (b_id, name_val, spec_key), group_rows in sorted_groups:
    barang_num = f"{SUB_CODE}-{seq_counter:04d}"
    lot_num = f"LOT-0001-26-{barang_num}"
    first_img = next((str(r.get("image_url") or r.get("image_url_unit")).strip() for _, r in enumerate(group_rows) if pd.notna(r.get("image_url") or r.get("image_url_unit")) and str(r.get("image_url") or r.get("image_url_unit")).strip()), None)
    c_ats = [str(r.get("created_at") or r.get("created_at_unit")) for r in group_rows if pd.notna(r.get("created_at") or r.get("created_at_unit")) and str(r.get("created_at") or r.get("created_at_unit")).strip()]
    u_ats = [str(r.get("updated_at") or r.get("updated_at_unit")) for r in group_rows if pd.notna(r.get("updated_at") or r.get("updated_at_unit")) and str(r.get("updated_at") or r.get("updated_at_unit")).strip()]
    org_id = resolve_organizer_id(group_rows[0].get("organizer"))

    sub_barangs.append({
        "id": seq_counter, "number": barang_num, "subcategory_id": sub_map[SUB_CODE],
        "brand_id": b_id, "uom_id": 1, "name": name_val, "specification": None,
        "min_stock_threshold": None, "image_url": first_img, "last_restock_at": None,
        "created_at": min(c_ats) if c_ats else now_str, "updated_at": max(u_ats) if u_ats else now_str
    })
    sub_lots.append({
        "id": seq_counter, "number": lot_num, "barang_id": seq_counter, "organizer_id": int(org_id),
        "vendor_id": None, "legacy_vendor_id": None, "location_id": None, "initial_quantity": None,
        "current_quantity": None, "po_number": None, "date_of_receipt": now_str, "unit_price": None,
        "image_url": first_img, "burden": None, "project_id": None, "created_at": now_str, "updated_at": now_str
    })
    for r in group_rows: unit_lot_map[str(r["id"]).strip()] = seq_counter
    seq_counter += 1

df_sub_barangs = pd.DataFrame(sub_barangs)
df_sub_lots = pd.DataFrame(sub_lots)
df_sub_barangs.to_csv(STAGING_BARANGS_DIR / f"{SUB_CODE}.csv", index=False)
df_sub_lots.to_csv(STAGING_LOTS_DIR / f"{SUB_CODE}.csv", index=False)
for idx in range(len(df_units)):
    ams_id = str(df_units.at[idx, "ams_asset_id"]).strip()
    if ams_id in unit_lot_map: df_units.at[idx, "lot_id"] = unit_lot_map[ams_id]
for c in ["id", "lot_id", "location_id", "project_id", "vendor_id"]:
    if c in df_units.columns: df_units[c] = pd.to_numeric(df_units[c], errors="coerce").astype("Int64")
df_units.to_csv(STAGING_UNITS_DIR / f"{SUB_CODE}.csv", index=False)
df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()
assert len(df_sub_barangs) == 2, f"Expected 2 barangs, got {len(df_sub_barangs)}"
print(f"\n[SUCCESS] Consolidated ACC-WB from 3 barangs down to 2 barangs.")
display(df_sub_barangs[["id", "number", "brand_id", "name", "specification"]])


Consolidating ACC-WB...
 Master Sync Complete: 1,443 barangs, 1,443 lots, 6,943 units.

[SUCCESS] Consolidated ACC-WB from 3 barangs down to 2 barangs.


,id,number,brand_id,name,specification
0,1,ACC-WB-0001,123,Whiteboard,None
1,2,ACC-WB-0002,25,Panaboard,None


---
## 14. Formatting Polish: ACC-PK & ACC-STB (Title Case)
### Formatting Rules applied:
1. **ACC-PK**: Name  -> .
2. **ACC-STB**: Name  -> .

In [15]:
# =====================================================================
# Formatting Polish: ACC-PK & ACC-STB (Title Case Names)
# =====================================================================
df_pk = pd.read_csv(STAGING_BARANGS_DIR / "ACC-PK.csv", dtype=str)
df_pk["name"] = df_pk["name"].apply(lambda x: str(x).strip().title() if pd.notna(x) else x)
df_pk.to_csv(STAGING_BARANGS_DIR / "ACC-PK.csv", index=False)

df_stb = pd.read_csv(STAGING_BARANGS_DIR / "ACC-STB.csv", dtype=str)
df_stb["name"] = df_stb["name"].apply(lambda x: str(x).strip().title() if pd.notna(x) else x)
df_stb.to_csv(STAGING_BARANGS_DIR / "ACC-STB.csv", index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()
print("\n[SUCCESS] Formatted ACC-PK and ACC-STB to Title Case.")
display(df_pk[["number", "name"]])
display(df_stb[["number", "name"]])


 Master Sync Complete: 1,443 barangs, 1,443 lots, 6,943 units.

[SUCCESS] Formatted ACC-PK and ACC-STB to Title Case.


,number,name
0,ACC-PK-0001,Pot Kecil


,number,name
0,ACC-STB-0001,Scaffolding
1,ACC-STB-0002,Storage Box


---
## 15. Subcategory ACC-GPS: GPS Consolidation
### Normalization Rules applied:
1. Remove  from  name ( -> ) as it is redundant with brand (DJI, Brand ID 90).

In [16]:
# =====================================================================
# Subcategory ACC-GPS: Remove Redundant Brand Name
# =====================================================================
df_gps = pd.read_csv(STAGING_BARANGS_DIR / "ACC-GPS.csv", dtype=str)
df_gps["name"] = df_gps["name"].replace({"DJI D-RTK 3": "D-RTK 3"})
df_gps.to_csv(STAGING_BARANGS_DIR / "ACC-GPS.csv", index=False)

df_all_barangs, df_all_lots, df_all_units = sync_consolidated_masters()
print("\n[SUCCESS] Updated ACC-GPS-0001 name to D-RTK 3.")
display(df_gps[["number", "brand_id", "name", "specification"]])


 Master Sync Complete: 1,443 barangs, 1,443 lots, 6,943 units.

[SUCCESS] Updated ACC-GPS-0001 name to D-RTK 3.


,number,brand_id,name,specification
0,ACC-GPS-0001,90,D-RTK 3,Multifunctional Station + Tripod Station
